# Advanced Real-World Data Science Project

This notebook demonstrates an end-to-end advanced classification workflow using a synthetic customer-retention-style dataset. It covers a baseline, preprocessing pipeline, validation, threshold analysis, permutation importance, and artifact metadata.

The data is synthetic and for education only. It should not be used for real customer decisions.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    accuracy_score, confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

## 1. Generate synthetic data

In [ ]:
X_array, y = make_classification(
    n_samples=1800,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.82, 0.18],
    class_sep=0.9,
    random_state=RANDOM_STATE,
)
feature_names = [f"feature_{i}" for i in range(1, 11)]
X = pd.DataFrame(X_array, columns=feature_names)
# Add a few missing values to demonstrate pipeline-based imputation.
rng = np.random.default_rng(RANDOM_STATE)
for col in ["feature_2", "feature_7"]:
    idx = rng.choice(X.index, size=25, replace=False)
    X.loc[idx, col] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Positive rate (train):", round(float(np.mean(y_train)), 3))

## 2. Establish a dummy baseline

In [ ]:
baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DummyClassifier(strategy="prior")),
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]
print("Baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Baseline F1:", round(f1_score(y_test, baseline_pred, zero_division=0), 3))
print("Baseline ROC-AUC:", round(roc_auc_score(y_test, baseline_prob), 3))

## 3. Train a preprocessing and classification pipeline

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    )),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

model_metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, prob)),
}
model_metrics

## 4. Confusion matrix and classification report

In [ ]:
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))

## 5. Cross-validation on training data

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring=["precision", "recall", "f1", "roc_auc"],
)
cv_summary = pd.DataFrame({
    metric: {
        "mean": cv_results[f"test_{metric}"].mean(),
        "std": cv_results[f"test_{metric}"].std(),
    }
    for metric in ["precision", "recall", "f1", "roc_auc"]
}).T
cv_summary

## 6. Compare decision thresholds

In [ ]:
threshold_rows = []
for threshold in [0.25, 0.50, 0.75]:
    threshold_pred = (prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, threshold_pred, zero_division=0),
        "recall": recall_score(y_test, threshold_pred, zero_division=0),
        "f1": f1_score(y_test, threshold_pred, zero_division=0),
        "predicted_positives": int(threshold_pred.sum()),
    })
threshold_table = pd.DataFrame(threshold_rows)
threshold_table

## 7. Permutation importance

In [ ]:
importance = permutation_importance(
    model, X_test, y_test, scoring="f1", n_repeats=8, random_state=RANDOM_STATE
)
importance_table = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)
importance_table

In [ ]:
importance_table.sort_values("importance_mean").plot(
    x="feature", y="importance_mean", kind="barh", legend=False,
    title="Permutation Importance (F1 scoring)"
)
plt.xlabel("Mean decrease in F1 after shuffling")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 8. Save model, metrics, and metadata

In [ ]:
joblib.dump(model, ARTIFACT_DIR / "model.joblib")  # Only load trusted artifacts.

pd.DataFrame([model_metrics]).to_csv(ARTIFACT_DIR / "metrics.csv", index=False)
metadata = {
    "model_name": "advanced-classification-demo",
    "model_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "features": list(X_train.columns),
    "random_state": RANDOM_STATE,
    "test_metrics": model_metrics,
    "intended_use": "Educational demonstration only",
    "limitations": [
        "Synthetic data does not represent a real business population.",
        "Metrics may not transfer to real-world data.",
        "Feature importance is not causal evidence."
    ],
}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved artifacts to:", ARTIFACT_DIR.resolve())

## Key takeaways

- Start with a baseline and compare it using appropriate metrics.
- Keep preprocessing inside the pipeline to reduce leakage.
- Use cross-validation for model selection and the final test set for final evaluation.
- Threshold choice depends on error costs and operational capacity.
- Interpretability methods are diagnostic, not proof of causation.
- Document the model's intended use, versions, metrics, and limitations.